<a href="https://colab.research.google.com/github/rajiv-ranjan/cds-mini-projects/blob/Archana/M6_NB_MiniProject_1_Medical_Q%26A_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification Program in Computational Data Science
## A programme by IISc and TalentSprint
### Mini-Project: Medical Q&A using GPT2

## Learning Objectives

At the end of the experiment, you will be able to:

* perform data preprocessing, EDA and feature extraction on the Medical Q&A dataset
* load a pre-trained tokenizer
* finetune a GPT-2 language model for medical question-answering

## Dataset Description

The dataset used in this project is the *Medical Question Answering Dataset* ([MedQuAD](https://github.com/abachaa/MedQuAD/tree/master)). It includes medical question-answer pairs along with additional information, such as the question type, the question *focus*, its UMLS(Unified Medical Language System) details like - Concept Unique Identifier(*CUI*) and Semantic *Type* and *Group*.

To know more about this data's collection, and construction method, refer to this [paper](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-019-3119-4).

The data is extracted and is in CSV format with below features:

- **Focus**: the question focus
- **CUI**: concept unique identifier
- **SemanticType**
- **SemanticGroup**
- **Question**
- **Answer**

## Part-A: Grading = 10 Points

## Information

Healthcare professionals often have to refer to medical literature and documents while seeking answers to medical queries. Medical databases or search engines are powerful resources of upto date medical knowledge. However, the existing documentation is large and makes it difficult for professionals to retrieve answers quickly in a clinical setting. The problem with search engines and informative retrieval engines is that these systems return a list of documents rather than answers. Instead, healthcare professionals can use question answering systems to retrieve short sentences or paragraphs in response to medical queries. Such systems have the biggest advantage of generating answers and providing hints in a few seconds.

### Problem Statement

Fine-tune gpt2 model on medical-question-answering-dataset for performing response generation for medical queries.

Please refer to ***M6 Assignment-1 Fine-tune GPT2*** to get familiar with how to load pre-trained gpt2 tokenizer and model.

### Import required packages

In [ ]:
!pip -q install -U accelerate
!pip -q install -U transformers
!pip -q install torch

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#@title Download the dataset
!wget -q https://cdn.iisc.talentsprint.com/AIandMLOps/MiniProjects/Datasets/MedQuAD.csv
!ls | grep ".csv"

**Exercise 1: Read the MedQuAD.csv dataset**

**Hint:** pd.read_csv()

In [ ]:
df = pd.read_csv("MedQuAD.csv")
df.shape

In [ ]:
df.head()

### Pre-processing and EDA

**Exercise 2: Perform below operations on the dataset [0.5 Mark]**

- Handle missing values
- Remove duplicates from data considering `Question` and `Answer` columns

- **Handle missing values**

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Drop missing values
df.dropna(subset=["Answer"], inplace=True)

In [ ]:
df.isnull().sum()

- **Remove duplicates from data considering `Question` and `Answer` columns**

In [ ]:
# Check duplicates
df.duplicated(subset=['Question', 'Answer']).sum()

In [ ]:
# Drop duplicates
df.drop_duplicates(subset=['Question', 'Answer'], inplace=True)

In [ ]:
# Check duplicates
df.duplicated(subset=['Question', 'Answer']).sum()

**Exercise 3: Display the category name, and the number of records belonging to top 100 categories of `Focus` column [1 Mark]**

In [ ]:
# YOUR CODE HERE
categoryName = df['Focus'].value_counts().index
categoryName

In [ ]:
# Top 100 Focus categories names
df['Focus'].value_counts().head(100)

### Create Training and Validation set

**Exercise 4: Create training and validation set [2 Marks]**

- Consider 4 samples per `Focus` category, for each top 100 categories, from the dataset (It will give 400 samples for training)

- Consider 1 sample per `Focus` category (different from training set), for each top 100 categories, from the dataset (It will give 100 samples for validation)

In [ ]:
# create list of top 100 categories
top_Focus = df['Focus'].value_counts().head(100).index.to_list()
top_Focus

In [ ]:
# YOUR CODE HERE
train_samples = []
val_samples = []

for focus in top_Focus:
    focus_df = df[df['Focus'] == focus]

    focus_df = focus_df.sample(n=5, random_state=42)
    train_samples.append(focus_df.iloc[:4])
    val_samples.append(focus_df.iloc[4:5])

train_df = pd.concat(train_samples).reset_index(drop=True)
val_df = pd.concat(val_samples).reset_index(drop=True)

print(len(train_df))
print(len(val_df))


### Pre-process `Question` and `Answer` text

**Exercise 5: Perform below tasks: [1.5 Marks]**

- Combine `Question` and `Answer` for train and validation data as shown below:
    - sequence = *'\<question\>' + question-text + '\<answer\>' + answer-text*

- Join the combined text using '\n' into a single string for training and validation separately

- Save the training and validation strings as separate text files

- **Combine Question and Answer for train and val data**

In [ ]:
train_df['text'] = '<question>' + train_df['Question'] + '<answer>' + train_df['Answer']
train_df.head()

In [ ]:
print(train_df['text'][0])

In [ ]:
val_df['text'] = '<question>' + val_df['Question'] + '<answer>' + val_df['Answer']
val_df.head()

- **Join the combined text using '\n' into a single string for training and validation separately**

In [ ]:
train_combined_text = '\n'.join(train_df['text'].to_list())
val_combined_text = '\n'.join(val_df['text'].to_list())

print(len(train_combined_text))
print(len(val_combined_text))

- **Save the training and validation strings as text files**

In [ ]:
with open('train_data.txt', 'w') as f:
    f.write(train_combined_text)

with open('val_data.txt', 'w') as f:
    f.write(val_combined_text)

**Exercise 6: Load pre-trained GPT2Tokenizer [0.5 Mark]**

- Use checkpoint = "gpt2"

In [ ]:
checkpoint = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(checkpoint)


**Exercise 7: Tokenize train and validation data and form TextDataset objects [0.5 Mark]**

- Use the loaded pre-trained tokenizer
- Use training and validation data saved in text files

In [ ]:
# Tokenize train text
train_dataset = TextDataset(tokenizer=tokenizer, file_path="train_data.txt", block_size=128)

# Tokenize validation text
val_dataset = TextDataset(tokenizer=tokenizer, file_path="val_data.txt", block_size=128)

In [ ]:
print(len(train_dataset))
print(len(val_dataset))

In [ ]:
# Batch-size
train_dataset[0].shape, val_dataset[0].shape

**Exercise 8: Create a DataCollator object [0.5 Mark]**

In [ ]:
# Create a Data collator object
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt")

**Exercise 9: Load pre-trained GPT2LMHeadModel [0.5 Mark]**

In [ ]:
# Set up the model
model = GPT2LMHeadModel.from_pretrained(checkpoint)

**Exercise 10: Fine-tune GPT2 Model [1 Mark]**

- Specify training arguments and create a TrainingArguments object (Use 30 epochs)

- Train a GPT-2 model using the provided training arguments

- Save the resulting trained model and tokenizer to a specified output directory

In [ ]:
# Set up the training arguments

model_output_path = "/content/gpt_model"

training_args = TrainingArguments(
    output_dir = model_output_path,
    overwrite_output_dir = True,
    per_device_train_batch_size = 4, # try with 2
    per_device_eval_batch_size = 4,  #  try with 2
    num_train_epochs = 30,
    save_steps = 1_000,
    save_total_limit = 2,
    logging_dir = './logs',
    )

In [ ]:
# Train the model
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
)

trainer.train()

# Save the model
trainer.save_model(model_output_path)

# Save the tokenizer
tokenizer.save_pretrained(model_output_path)

**Exercise 11: Test Model with user input prompts [1 Mark]**

- Create `generate_response()` function that takes a trained *model*, *tokenizer*, and a *prompt* string as input and generates a response using the GPT-2 model

- Test it with some user input prompts

In [ ]:
# YOUR CODE HERE
def generate_response(model, tokenizer, prompt, max_length=100):

    input_ids = tokenizer.encode(prompt, return_tensors="pt")      # 'pt' for returning pytorch tensor

    # Create the attention mask and pad token id
    attention_mask = torch.ones_like(input_ids)
    pad_token_id = tokenizer.eos_token_id

    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        attention_mask=attention_mask,
        pad_token_id=pad_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Load the fine-tuned model and tokenizer

my_model = GPT2LMHeadModel.from_pretrained(model_output_path)
my_tokenizer = GPT2Tokenizer.from_pretrained(model_output_path)

In [ ]:
# Response from model

prompt = "Who is at risk for Stroke?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with given prompt 1

prompt = "How to prevent Skin Cancer ?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with given prompt 2

prompt = "What are the treatment for cancer?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

**Exercise 12: Compare the performance of a *GPT2 model* with the *GPT2 model fine-tuned* on MedQuAD data [1 Mark]**

- Load another pre-trained GPT2LMHeadModel and do not fine-tune it

- To generate response using the untuned model, pass it as a parameter to `generate_response()` function

- Test both models (fine-tuned and untuned) with below user input prompts:

    - "What precautions to take for a healthy life?"
    - "What to do after being diagnosed with cancer?"
    - "What to do when feeling sick?"

In [ ]:
# Load a pre-trained GPT2 model, do not finetune it with MedQuAD data

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [ ]:
# Testing with finetuned model: prompt 1

prompt = "What are the symtoms of tuberculosis?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with untuned model: prompt 1

prompt = "What are the symtoms of tuberculosis?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with finetuned model: prompt 2

prompt = "What causes dry eye?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with untuned model: prompt 2

prompt = "What causes dry eye?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with finetuned model: prompt 3

prompt = "What is the treatment for ligament injury?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

In [ ]:
# Testing with untuned model: prompt 3

prompt = "What is the treatment for ligament injury?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)